# NL2SQL confidence under perturbation: Checkpoint 2 runs (Colab)

Runs every CP2 experiment (E1 to E6) from `configs/cp2_plan.yaml` on one GPU.

**Before you start**
1. *Runtime > Change runtime type*: **A100 GPU**, **High-RAM** on. (A G4 runtime also works; L4 and T4 cannot fit the 14B model.)
2. Put `nl2sql_confidence_code.zip` in Google Drive at `MyDrive/nl2sql_cp2/` (or upload it when cell 3 asks).
3. *Runtime > Run all*. Keep this tab open.

**If the runtime disconnects:** reconnect and *Run all* again. Results live in Drive and every stage resumes where it stopped (at most one batch is redone).

Expected time on an A100: roughly 4 to 6 h for the full plan, preceded by a ~5 min smoke test. Cell 9 shows progress at any time.

In [ ]:
# 1. Settings
DRIVE_DIR   = "/content/drive/MyDrive/nl2sql_cp2"     # results + caches live here
CODE_ZIP    = "nl2sql_confidence_code.zip"            # code bundle, inside DRIVE_DIR
RUN_SMOKE   = True     # 5-minute end-to-end check on the 0.5B model before the real run
RUN_FULL    = True
ONLY        = ""       # e.g. "0.5b,1.5b" to run a subset of E1 models; "" = all
REPO        = "/content/nl2sql_confidence"

In [ ]:
# 2. GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
print(f"{props.name}: {vram:.0f} GiB, bf16 supported: {torch.cuda.is_bf16_supported()}")
x = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16); (x @ x).sum().item()
if vram < 38 and not ONLY:
    print("WARNING: under 38 GiB. The 14B model will not fit; set ONLY to skip it, e.g. "
          "ONLY = '0.5b,1.5b,3b,7b,deepseek'")

In [ ]:
# 3. Mount Drive and unpack the code
import os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
zip_path = f"{DRIVE_DIR}/{CODE_ZIP}"
if not os.path.exists(zip_path):
    from google.colab import files
    print(f"{zip_path} not found: choose {CODE_ZIP} from your computer")
    up = files.upload()
    shutil.move(next(iter(up)), zip_path)
shutil.rmtree(REPO, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
print(open(f"{REPO}/colab/BUNDLE_INFO.txt").read())

In [ ]:
# 4. Install pinned libraries (Colab's own PyTorch is kept)
%cd {REPO}
!pip install -q -r colab/requirements-colab.txt
!pip install -q -e . --no-deps
import importlib, transformers, omegaconf, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| omegaconf", omegaconf.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# 5. Data caches: Dr.Spider (168 MB) and test-suite DBs (1.27 GB) are kept in Drive
#    so a reconnect does not download them again.
CACHE = f"{DRIVE_DIR}/cache"; os.makedirs(CACHE, exist_ok=True); os.makedirs(f"{REPO}/data/raw", exist_ok=True)
for name in ["drspider_data.tar.gz", "testsuitedatabases.zip"]:
    src, dst = f"{CACHE}/{name}", f"{REPO}/data/raw/{name}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print("restored", name, "from Drive cache")

In [ ]:
# 6. Unit tests (fast)
!python -m pytest -q

In [ ]:
# 7. Smoke test: every stage on 0.5B with ~50 questions (about 5 min)
if RUN_SMOKE:
    !python scripts/run_cp2.py --plan configs/cp2_plan_smoke.yaml --results-root {DRIVE_DIR}/smoke_results
    # save the downloaded archives to the Drive cache for later reconnects
    for name in ["drspider_data.tar.gz", "testsuitedatabases.zip"]:
        if os.path.exists(f"{REPO}/data/raw/{name}") and not os.path.exists(f"{CACHE}/{name}"):
            shutil.copy(f"{REPO}/data/raw/{name}", f"{CACHE}/{name}"); print("cached", name)
    print(open(f"{DRIVE_DIR}/smoke_results/smoke_summary/exec_ok/CP2_SUMMARY.md").read()[:1500])

In [ ]:
# 8. Full Checkpoint 2 run (hours). Safe to re-run after a disconnect: it resumes.
if RUN_FULL:
    only = f"--only {ONLY}" if ONLY else ""
    !python scripts/run_cp2.py --results-root {DRIVE_DIR}/results {only}

In [ ]:
# 9. Progress table (can be run any time, e.g. after a reconnect)
!python scripts/run_cp2.py --results-root {DRIVE_DIR}/results --status

In [ ]:
# 10. Small results bundle for the write-up (summaries, metrics, figures, CSVs, logs;
#     excludes hidden-state features and per-token arrays). Saved to Drive.
import glob
bundle = f"{DRIVE_DIR}/cp2_results_bundle.zip"
root = f"{DRIVE_DIR}/results"
keep = (".md", ".json", ".csv", ".png", ".yaml", ".log")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{root}/**/*", recursive=True):
        if path.endswith(keep) and "/features/" not in path:
            z.write(path, os.path.relpath(path, DRIVE_DIR))
print(f"{bundle}: {os.path.getsize(bundle) / 2**20:.1f} MiB")
s = f"{root}/cp2_summary/exec_ok/CP2_SUMMARY.md"
if os.path.exists(s): print(open(s).read())